<a href="https://colab.research.google.com/github/Jiazzzzzzz/z/blob/main/Hurst_exponent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [103]:
#@title { vertical-output: true }
import numpy as np
import pandas as pd
from scipy.stats import linregress


class HurstExponent:
    """
    Hurst Exponent Estimator using:

        H = 1/2 * slope( log(Var(tau)) vs log(tau) )

    Interpretation:
    ----------------
    H < 0.5  -> Mean-reverting / anti-persistent
    H = 0.5  -> Random walk
    H > 0.5  -> Trending / persistent
    """

    def __init__(self, max_lag=100):
        self.max_lag = max_lag

    def compute(self, prices):
        """
        Parameters
        ----------
        prices : array-like
            Price series (list, numpy array, pandas Series)

        Returns
        -------
        dict
            {
                'hurst': float,
                'slope': float,
                'r_squared': float
            }
        """

        prices = np.asarray(prices)

        if len(prices) < self.max_lag:
            raise ValueError(
                f"Price series length must be > max_lag ({self.max_lag})"
            )

        lags = range(2, self.max_lag)

        tau = []

        # Variance of lagged differences
        for lag in lags:
            diff = prices[lag:] - prices[:-lag]
            tau.append(np.var(diff))

        tau = np.array(tau)

        # Remove zeros to avoid log issues
        valid = tau > 0

        log_lags = np.log(np.array(list(lags))[valid])
        log_tau = np.log(tau[valid])

        # Linear regression
        slope, intercept, r_value, p_value, std_err = linregress(
            log_lags,
            log_tau
        )

        hurst = slope / 2

        return {
            "hurst": hurst,
            "slope": slope,
            "r_squared": r_value**2
        }




In [104]:
#@title { vertical-output: true }
import yfinance as yf
import numpy as np
import pandas as pd
from scipy.stats import linregress


class HurstExponent:

    def __init__(self, max_lag=20):
        self.max_lag = max_lag

    def compute(self, series):

        series = np.asarray(series)

        valid_lags = []
        tau = []

        for lag in range(2, self.max_lag):

            diff = series[lag:] - series[:-lag]

            std = np.std(diff)

            if std > 0 and not np.isnan(std):

                tau.append(std)
                valid_lags.append(lag)

        tau = np.array(tau)
        valid_lags = np.array(valid_lags)

        log_lags = np.log(valid_lags)
        log_tau = np.log(tau)

        slope, intercept, r_value, p_value, std_err = linregress(
            log_lags,
            log_tau
        )

        return {
            "hurst": slope,
            "r_squared": r_value**2
        }


# =====================================================
# CUSTOM SETTINGS
# =====================================================

TICKER = "GC=F"

# LOOKBACK SYSTEM
LOOKBACK_PERIOD = "1d"

# INTERVAL
INTERVAL = "5m"

# HURST SETTINGS
MAX_LAG = 36


# =====================================================
# DOWNLOAD DATA
# =====================================================

df = yf.download(
    TICKER,
    period=LOOKBACK_PERIOD,
    interval=INTERVAL,
    auto_adjust=True,
    progress=False
)

# =====================================================
# CLEAN CLOSE PRICE
# =====================================================

close_prices = df["Close"].squeeze()
close_prices = pd.Series(close_prices).dropna()

# =====================================================
# LOG RETURNS
# =====================================================

log_returns = np.log(
    close_prices / close_prices.shift(1)
).dropna()

# =====================================================
# CUMULATIVE PROCESS
# =====================================================

series = np.cumsum(log_returns)

# =====================================================
# HURST EXPONENT
# =====================================================

model = HurstExponent(max_lag=MAX_LAG)

result = model.compute(series)

# =====================================================
# OUTPUT
# =====================================================

print("\n========== HURST EXPONENT ANALYSIS ==========")

print(f"Ticker          : {TICKER}")
print(f"Lookback Period : {LOOKBACK_PERIOD}")
print(f"Interval        : {INTERVAL}")
print(f"Max Lag         : {MAX_LAG}")
print(f"Data Points     : {len(series)}")

print("\nResults")
print("------------------------------------------------")

print(f"Hurst           : {result['hurst']:.4f}")
print(f"R²              : {result['r_squared']:.4f}")

# =====================================================
# INTERPRETATION
# =====================================================

H = result["hurst"]

print("\nInterpretation")
print("------------------------------------------------")

if H < 0.45:
    print("Mean-Reverting / Anti-Persistent")

elif H > 0.55:
    print("Trending / Persistent")

else:
    print("Efficient / Random Walk")


========== HURST EXPONENT ANALYSIS ==========
Ticker          : GC=F
Lookback Period : 1d
Interval        : 5m
Max Lag         : 36
Data Points     : 180

Results
------------------------------------------------
Hurst           : 0.1903
R²              : 0.4801

Interpretation
------------------------------------------------
Mean-Reverting / Anti-Persistent


In [ ]:
#@title { vertical-output: true }
# =====================================================
# ROLLING HURST EXPONENT VISUALIZATION
# =====================================================

import matplotlib.pyplot as plt


# =====================================================
# ROLLING HURST FUNCTION
# =====================================================

def rolling_hurst(series, window=300, max_lag=20):

    hurst_values = []
    dates = []

    series = np.asarray(series)

    for i in range(window, len(series)):

        chunk = series[i - window:i]

        valid_lags = []
        tau = []

        for lag in range(2, max_lag):

            diff = chunk[lag:] - chunk[:-lag]

            std = np.std(diff)

            if std > 0 and not np.isnan(std):

                tau.append(std)
                valid_lags.append(lag)

        # Skip unstable windows
        if len(tau) < 5:
            continue

        tau = np.array(tau)
        valid_lags = np.array(valid_lags)

        log_lags = np.log(valid_lags)
        log_tau = np.log(tau)

        slope, intercept, r_value, p_value, std_err = linregress(
            log_lags,
            log_tau
        )

        hurst_values.append(slope)

        dates.append(close_prices.index[i])

    return pd.DataFrame({
        "Datetime": dates,
        "Hurst": hurst_values
    })


# =====================================================
# CALCULATE ROLLING HURST
# =====================================================

rolling_window = 300

hurst_df = rolling_hurst(
    series=series,
    window=rolling_window,
    max_lag=MAX_LAG
)


# =====================================================
# VISUALIZATION
# =====================================================

plt.figure(figsize=(16, 7))

# Rolling Hurst line
plt.plot(
    hurst_df["Datetime"],
    hurst_df["Hurst"],
    linewidth=1.5,
    label="Rolling Hurst"
)

# Random walk baseline
plt.axhline(
    y=0.50,
    linestyle="--",
    linewidth=1.5,
    label="Random Walk (0.50)"
)

# Mean reverting zone
plt.axhline(
    y=0.45,
    linestyle=":",
    linewidth=1.2,
    label="Mean Reverting Zone"
)

# Trending zone
plt.axhline(
    y=0.55,
    linestyle=":",
    linewidth=1.2,
    label="Trending Zone"
)

# Labels
plt.title(
    f"Rolling Hurst Exponent ({TICKER})",
    fontsize=16
)

plt.xlabel("Datetime")
plt.ylabel("Hurst Exponent")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
#custom period
#@title { vertical-output: true }
import yfinance as yf
import numpy as np
import pandas as pd
from scipy.stats import linregress


class HurstExponent:

    def __init__(self, max_lag=20):
        self.max_lag = max_lag

    def compute(self, series):

        series = np.asarray(series)

        valid_lags = []
        tau = []

        for lag in range(2, self.max_lag):

            diff = series[lag:] - series[:-lag]

            std = np.std(diff)

            if std > 0 and not np.isnan(std):

                tau.append(std)
                valid_lags.append(lag)

        tau = np.array(tau)
        valid_lags = np.array(valid_lags)

        log_lags = np.log(valid_lags)
        log_tau = np.log(tau)

        slope, intercept, r_value, p_value, std_err = linregress(
            log_lags,
            log_tau
        )

        return {
            "hurst": slope,
            "r_squared": r_value**2
        }


# =====================================================
# CUSTOM SETTINGS
# =====================================================

TICKER = "GC=F"

START_DATE = "2026-05-01"
END_DATE   = "2026-05-10"

INTERVAL = "5m"

MAX_LAG = 30


# =====================================================
# DOWNLOAD DATA
# =====================================================

df = yf.download(
    TICKER,
    start=START_DATE,
    end=END_DATE,
    interval=INTERVAL,
    auto_adjust=True,
    progress=False
)

# =====================================================
# CLEAN CLOSE PRICE
# =====================================================

close_prices = df["Close"].squeeze()
close_prices = pd.Series(close_prices).dropna()

# =====================================================
# LOG RETURNS
# =====================================================

log_returns = np.log(
    close_prices / close_prices.shift(1)
).dropna()

# =====================================================
# CUMULATIVE PROCESS
# =====================================================

series = np.cumsum(log_returns)

# =====================================================
# HURST EXPONENT
# =====================================================

model = HurstExponent(max_lag=MAX_LAG)

result = model.compute(series)

# =====================================================
# OUTPUT
# =====================================================

print("\n========== HURST EXPONENT ANALYSIS ==========")

print(f"Ticker        : {TICKER}")
print(f"Start Date    : {START_DATE}")
print(f"End Date      : {END_DATE}")
print(f"Interval      : {INTERVAL}")
print(f"Data Points   : {len(series)}")

print("\nResults")
print("--------------------------------------------")
print(f"Hurst         : {result['hurst']:.4f}")
print(f"R²            : {result['r_squared']:.4f}")

# =====================================================
# INTERPRETATION
# =====================================================

H = result["hurst"]

print("\nInterpretation")
print("--------------------------------------------")

if H < 0.45:
    print("Mean-Reverting / Anti-Persistent")

elif H > 0.55:
    print("Trending / Persistent")

else:
    print("Efficient / Random Walk")